# Star Classification — Neural Network with RandomizedSearchCV Hyperparameter Tuning

This notebook wraps the Keras model using **SciKeras** (`KerasClassifier`) so it is fully compatible with sklearn's `RandomizedSearchCV`. The search targets the parameters most responsible for overfitting:

| Parameter | What it controls |
|---|---|
| `learning_rate` | Optimiser step size |
| `dropout_rate` | Fraction of neurons randomly zeroed per layer |
| `l2_lambda` | L2 weight-penalty strength |
| `neurons` | Hidden-layer width |
| `batch_size` | Mini-batch size during training |

## 1 · Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report

from category_encoders import BinaryEncoder, TargetEncoder

import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense, Dropout
from keras.optimizers import Adam
from keras.regularizers import l2
from keras.callbacks import EarlyStopping
from keras.utils import to_categorical

from scikeras.wrappers import KerasClassifier

import warnings
warnings.filterwarnings('ignore')

tf.random.set_seed(42)
np.random.seed(42)

## 2 · Load & explore data

In [ ]:
data = pd.read_csv('Stars.csv')
print(data.shape)
data.head()

In [ ]:
data.info()

In [ ]:
data['Type'].value_counts()

## 3 · Feature engineering & encoding

In [ ]:
# Binary-encode Spectral_Class
bin_enc = BinaryEncoder(cols=['Spectral_Class'])
data_encoded = bin_enc.fit_transform(data)

# Target-encode Color (using Temperature as the target for encoding)
tgt_enc = TargetEncoder(cols=['Color'])
data_encoded[['Color']] = tgt_enc.fit_transform(
    data_encoded[['Color']], data['Temperature']
)

data_encoded.head()

## 4 · Train/test split & scaling

In [ ]:
X = data_encoded.drop(columns='Type')
y = data_encoded['Type'].values          # integer labels — SciKeras handles one-hot internally

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

robust_cols = ['Temperature', 'L', 'R']
standard_cols = ['A_M', 'Color']

preprocessor = ColumnTransformer(
    transformers=[
        ('robust', RobustScaler(), robust_cols),
        ('std',    StandardScaler(), standard_cols),
    ],
    remainder='passthrough'
)

X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled  = preprocessor.transform(X_test)

n_features = X_train_scaled.shape[1]
n_classes  = len(np.unique(y))
print(f"Features: {n_features}  |  Classes: {n_classes}")

## 5 · Model factory for SciKeras

`KerasClassifier` calls `build_fn` once per candidate configuration and passes **only the parameters you declare as keyword arguments** down to it — that is how the search injects different values each trial.

In [ ]:
def build_model(
    neurons      = 64,
    dropout_rate = 0.3,
    l2_lambda    = 0.001,
    learning_rate= 0.001,
    n_features_in_= 8,    # injected automatically by SciKeras after fit
    n_classes_   = 6,     # injected automatically by SciKeras after fit
):
    """Build a 3-hidden-layer network whose width halves every layer."""
    reg = l2(l2_lambda)

    model = Sequential([
        Dense(neurons,      activation='relu',
              kernel_regularizer=reg, input_shape=(n_features_in_,)),
        Dropout(dropout_rate),

        Dense(neurons // 2, activation='relu', kernel_regularizer=reg),
        Dropout(dropout_rate),

        Dense(neurons // 4, activation='relu', kernel_regularizer=reg),
        Dropout(max(dropout_rate - 0.1, 0.0)),  # slightly less in final hidden layer

        Dense(n_classes_, activation='softmax'),
    ])

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',   # SciKeras keeps labels as integers
        metrics=['accuracy'],
    )
    return model

## 6 · Wrap in KerasClassifier

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)

classifier = KerasClassifier(
    model          = build_model,
    epochs         = 100,
    batch_size     = 32,
    validation_split = 0.2,
    callbacks      = [early_stop],
    verbose        = 0,
)

## 7 · Define the search space

Parameter names must match the keyword arguments in `build_model` **exactly** (SciKeras routes them via `model__` prefix or direct kwargs).

In [ ]:
param_distributions = {
    'model__neurons':       [32, 64, 128, 256],
    'model__dropout_rate':  [0.1, 0.2, 0.3, 0.4, 0.5],
    'model__l2_lambda':     [1e-4, 5e-4, 1e-3, 5e-3, 1e-2],
    'model__learning_rate': [1e-4, 5e-4, 1e-3, 5e-3],
    'batch_size':           [16, 32, 64],
}

## 8 · Run RandomizedSearchCV

`n_iter=20` evaluates 20 random combinations with 3-fold cross-validation (60 model fits total). Increase `n_iter` for a broader search at the cost of time.

In [ ]:
random_search = RandomizedSearchCV(
    estimator          = classifier,
    param_distributions= param_distributions,
    n_iter             = 20,          # number of random candidates
    cv                 = 3,           # 3-fold CV
    scoring            = 'accuracy',
    n_jobs             = 1,           # set -1 to use all cores (may conflict with TF)
    random_state       = 42,
    verbose            = 1,
    return_train_score = True,
)

random_search.fit(X_train_scaled, y_train)

## 9 · Inspect results

In [ ]:
print("Best CV accuracy : ", round(random_search.best_score_, 4))
print("\nBest parameters  :")
for k, v in random_search.best_params_.items():
    print(f"  {k}: {v}")

In [ ]:
# Summarise all candidates sorted by mean CV accuracy
results_df = pd.DataFrame(random_search.cv_results_)
results_df = results_df.sort_values('mean_test_score', ascending=False)

cols_to_show = [
    'param_model__neurons', 'param_model__dropout_rate',
    'param_model__l2_lambda', 'param_model__learning_rate',
    'param_batch_size',
    'mean_train_score', 'mean_test_score', 'std_test_score'
]
results_df[cols_to_show].head(10).round(4)

### Overfit check
The gap between `mean_train_score` and `mean_test_score` quantifies overfitting. A well-tuned model should show a small gap (< ~3 %).

In [ ]:
results_df['overfit_gap'] = (
    results_df['mean_train_score'] - results_df['mean_test_score']
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Scatter: train vs. val accuracy ---
axes[0].scatter(
    results_df['mean_train_score'],
    results_df['mean_test_score'],
    c=results_df['overfit_gap'], cmap='RdYlGn_r', s=80, edgecolors='k'
)
lims = [0.6, 1.01]
axes[0].plot(lims, lims, 'k--', linewidth=1, label='No overfit line')
axes[0].set_xlim(lims); axes[0].set_ylim(lims)
axes[0].set_xlabel('Mean Train Accuracy (CV)')
axes[0].set_ylabel('Mean Val Accuracy (CV)')
axes[0].set_title('Train vs. Validation Accuracy')
axes[0].legend()

# --- Bar: overfit gap per candidate ---
top10 = results_df.head(10).reset_index(drop=True)
colors = ['green' if g < 0.03 else 'orange' if g < 0.06 else 'red'
          for g in top10['overfit_gap']]
axes[1].bar(range(len(top10)), top10['overfit_gap'], color=colors)
axes[1].axhline(0.03, color='green', linestyle='--', label='3 % threshold')
axes[1].set_xlabel('Candidate rank (0 = best CV accuracy)')
axes[1].set_ylabel('Overfit gap (train − val)')
axes[1].set_title('Overfitting Gap — Top 10 Candidates')
axes[1].legend()

plt.tight_layout()
plt.show()

## 10 · Retrain best model on full training set & evaluate on held-out test set

In [ ]:
best_params = random_search.best_params_

best_model = build_model(
    neurons       = best_params['model__neurons'],
    dropout_rate  = best_params['model__dropout_rate'],
    l2_lambda     = best_params['model__l2_lambda'],
    learning_rate = best_params['model__learning_rate'],
    n_features_in_= n_features,
    n_classes_    = n_classes,
)

history = best_model.fit(
    X_train_scaled, y_train,
    epochs           = 150,
    batch_size       = best_params['batch_size'],
    validation_split = 0.2,
    callbacks        = [EarlyStopping(monitor='val_loss', patience=12,
                                      restore_best_weights=True)],
    verbose          = 1,
)

In [ ]:
# Learning curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, title in zip(
    axes,
    [('loss', 'val_loss'), ('accuracy', 'val_accuracy')],
    ['Loss', 'Accuracy']
):
    ax.plot(history.history[metric[0]], label=f'Train {title}')
    ax.plot(history.history[metric[1]], label=f'Validation {title}')
    ax.set_title(f'{title} — Best Model')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(title)
    ax.legend()

plt.suptitle('Best Model Learning Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Final evaluation
train_preds = np.argmax(best_model.predict(X_train_scaled), axis=1)
test_preds  = np.argmax(best_model.predict(X_test_scaled),  axis=1)

star_types = ['Brown Dwarf', 'Red Dwarf', 'White Dwarf',
              'Main Sequence', 'Supergiant', 'Hypergiant']

print("=" * 55)
print("TRAINING SET")
print("=" * 55)
print(classification_report(y_train, train_preds, target_names=star_types))

print("=" * 55)
print("TEST SET")
print("=" * 55)
print(classification_report(y_test, test_preds, target_names=star_types))

In [ ]:
# Quick overfit summary
from sklearn.metrics import accuracy_score
train_acc = accuracy_score(y_train, train_preds)
test_acc  = accuracy_score(y_test,  test_preds)

print(f"Train accuracy : {train_acc:.4f}")
print(f"Test  accuracy : {test_acc:.4f}")
print(f"Overfit gap    : {train_acc - test_acc:.4f}")

if train_acc - test_acc < 0.05:
    print("✅  Gap < 5 % — model generalises well.")
else:
    print("⚠️  Gap ≥ 5 % — consider more regularisation.")

## 11 · Summary of tuned hyperparameters

The table below records every design decision made by the random search.

In [ ]:
summary = pd.DataFrame([
    {'Parameter':     'neurons',
     'Search space':  '[32, 64, 128, 256]',
     'Best value':    best_params['model__neurons'],
     'Why it matters': 'Wider → more capacity → higher overfit risk'},

    {'Parameter':     'dropout_rate',
     'Search space':  '[0.1 … 0.5]',
     'Best value':    best_params['model__dropout_rate'],
     'Why it matters': 'Higher dropout → stronger regularisation'},

    {'Parameter':     'l2_lambda',
     'Search space':  '[1e-4 … 1e-2]',
     'Best value':    best_params['model__l2_lambda'],
     'Why it matters': 'Penalises large weights; reduces variance'},

    {'Parameter':     'learning_rate',
     'Search space':  '[1e-4 … 5e-3]',
     'Best value':    best_params['model__learning_rate'],
     'Why it matters': 'Too high → diverge; too low → slow convergence'},

    {'Parameter':     'batch_size',
     'Search space':  '[16, 32, 64]',
     'Best value':    best_params['batch_size'],
     'Why it matters': 'Smaller batches add noise → implicit regularisation'},
])

summary.set_index('Parameter')